In [1]:
import os
import logging

# 1. Force C++ backend to only show FATAL errors
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

# 2. Silence Python's absl logging module before importing TensorFlow
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

# 3. Silence Python's standard logging for TensorFlow
logging.getLogger('tensorflow').setLevel(logging.FATAL)

import tensorflow as tf

In [2]:
import sys

REPO_NAME = "RefraScan"
GITHUB_USER = "KyziaPi"
BRANCH_NAME = "ResNet50-Train"   # <-- point this at your branch

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_PATH = os.path.join("/kaggle/working", REPO_NAME)

if not os.path.exists(REPO_NAME):
    print(f"Cloning branch '{BRANCH_NAME}' from {REPO_NAME}...")
    !env GIT_TERMINAL_PROMPT=0 git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists. Switching branch and pulling latest updates...")
    !cd {REPO_NAME} && env GIT_TERMINAL_PROMPT=0 git fetch --all && git checkout {BRANCH_NAME} && git pull origin {BRANCH_NAME}

if os.path.exists(REPO_PATH):
    if REPO_PATH not in sys.path:
        sys.path.append(REPO_PATH)

    import math
    import pandas as pd
    from tensorflow.keras.applications.resnet50 import preprocess_input

    try:
        from src.preprocessing import load_and_clean_data
        from src.cross_validation import run_cross_validation
        print("🚀 Success! Custom modules imported smoothly.")
    except ModuleNotFoundError as e:
        print(f"❌ Still failing. Current sys.path contains: {sys.path}")
        raise e

    print(f"Environment configured successfully! Working on branch: {BRANCH_NAME}")
else:
    print("❌ Error: Repository failed to clone.")

Cloning branch 'ResNet50-Train' from RefraScan...
Cloning into 'RefraScan'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 264 (delta 16), reused 40 (delta 13), pack-reused 219 (from 1)
Receiving objects: 100% (264/264), 1.19 MiB | 5.09 MiB/s, done.
Resolving deltas: 100% (118/118), done.
🚀 Success! Custom modules imported smoothly.
Environment configured successfully! Working on branch: ResNet50-Train


In [3]:
DATASET_DIR = '/kaggle/input/datasets/yerikaelainegueco/fundus-images-with-refractive-values' 
CSV_PATH = os.path.join(DATASET_DIR, 'RefraScan_dataset.csv')
IMG_DIR = os.path.join(DATASET_DIR, 'FundusImages')

df = load_and_clean_data(CSV_PATH, IMG_DIR)

df['classification_encoded'] = df['classification'].map(
    {'Emmetropia': 0, 'Myopia': 1, 'Hyperopia': 2}
)

results = run_cross_validation(
    df=df,
    model_name='resnet50',
    preprocess_input=preprocess_input,
    patient_col="ID",
    target_col="classification_encoded",
    n_splits=10,
    batch_size=16,
    epochs=30,
    learning_rate=0.0001,
    holdout_test_size=0.15,
)

Dataset Split: 866 samples for 10-Fold CV | 152 samples in Holdout Test Set (15%)

STARTING 10-FOLD CROSS VALIDATION


--- Fold 1/10 ---


I0000 00:00:1784972152.959277      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784972152.965433      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/30
 1/96 ━━━━━━━━━━━━━━━━━━━━ 24:32 15s/step - accuracy: 0.3125 - loss: 2.8524

I0000 00:00:1784972177.598549     140 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 502ms/step - accuracy: 0.3718 - loss: 1.0145
Epoch 1: val_loss improved from None to 0.35568, saving model to best_resnet50_fold_1.h5

Epoch 1: finished saving model to best_resnet50_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 71s 583ms/step - accuracy: 0.4310 - loss: 0.7892 - val_accuracy: 0.7083 - val_loss: 0.3557
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.4835 - loss: 0.5989
Epoch 2: val_loss improved from 0.35568 to 0.29316, saving model to best_resnet50_fold_1.h5

Epoch 2: finished saving model to best_resnet50_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 44s 460ms/step - accuracy: 0.5234 - loss: 0.5399 - val_accuracy: 0.7292 - val_loss: 0.2932
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - accuracy: 0.5856 - loss: 0.4748
Epoch 3: val_loss improved from 0.29316 to 0.25253, saving model to best_resnet50_fold_1.h5

Epoch 3: finished saving model to best_resnet50_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 43s 456ms/step - accuracy: 0.5664 - loss